### РОССИЙСКИЙ УНИВЕРСИТЕТ ДРУЖБЫ НАРОДОВ  
### Факультет искусственного интеллекта  
### Дисциплина: Инструменты обработки и визуализации данных  
## ОТЧЕТ ПО ЛАБОРАТОРНОЙ РАБОТЕ № 1
#### Студент:  Князев Арсений
#### Группа:   ЗФИбд-01-24  
#### Вариант № 7

In [ ]:
import sys

import mrjob

TEXT = "Story 7 HAWTHORNE.txt"

print("python:", sys.version.split()[0])



Шаг 1 - частота слов, оставляем те, что встретились 1 раз.

Шаг 2 - группируем их по длине.

In [ ]:
%%file MR_unique_words.py
# Вариант 7 - слова, которые встречаются в тексте ровно один раз

import json
from string import punctuation

from mrjob.job import MRJob
from mrjob.protocol import TextValueProtocol
from mrjob.step import MRStep


class MRUniqueWordsByLength(MRJob):
    # чтобы вывод был обычным текстом с табами, а не JSON
    OUTPUT_PROTOCOL = TextValueProtocol

    def steps(self):
        return [
            MRStep(mapper=self.mapper_words, reducer=self.reducer_only_unique),
            # маппер на втором шаге не нужен, mrjob сам передаст данные из первого редьюсера
            MRStep(reducer=self.reducer_group_by_length),
        ]

    # шаг 1: частота слов, оставляем только те, что встретились один раз

    def mapper_words(self, _, line):
        # "--" в тексте стоит между словами, поэтому меняем на пробел
        for token in line.lower().replace("--", " ").split():
            word = token.strip(punctuation)  # убираем знаки по краям слова
            if word:
                yield word, 1

    def reducer_only_unique(self, word, counts):
        total = 0
        for count in counts:
            total += count
            if total > 1:
                return  # слово уже не уникальное, дальше считать незачем
        yield len(word), word

    # шаг 2: группируем уникальные слова по длине

    def reducer_group_by_length(self, length, words):
        count = 0
        group = []
        for word in words:
            count += 1
            group.append(word)
        group.sort()
        yield None, f"{length}\t{count}\t{json.dumps(group, ensure_ascii=False)}"


if __name__ == "__main__":
    MRUniqueWordsByLength.run()

### Запуск

In [ ]:
!{sys.executable} MR_unique_words.py "{TEXT}" -q > result.txt

In [ ]:
# что получилось (ключи идут не по порядку: шаффл сортирует их как строки)
with open("result.txt", encoding="utf-8") as f:
    for line in f:
        length, count, words = line.rstrip("\n").split("\t")
        print(f"{length}\t{count}\t{words}")

### Результат

In [ ]:
import json

import pandas as pd

rows = []
with open("result.txt", encoding="utf-8") as f:
    for line in f:
        length, count, words_json = line.rstrip("\n").split("\t")
        rows.append((int(length), int(count), json.loads(words_json)))
rows.sort()

report = pd.DataFrame(
    [
        (length, count, ", ".join(words[:8]) + (", ..." if len(words) > 8 else ""))
        for length, count, words in rows
    ],
    columns=["длина слова", "кол-во слов", "слова (первые 8)"],
)
report

In [ ]:
# полный вывод в формате задания
for length, count, words in rows:
    print(f"{length}\t{count}\t[{', '.join(words)}]")